# Search Type Comparison: Hybrid vs Text vs Vector

## Purpose

This notebook compares the three search types supported by the project's FAISS+BM25 search index:

| Search Type | How It Works |
|-------------|-------------|
| **Text (BM25)** | Classic term-frequency matching on `filename` and `content` fields via `multi_match`. Excels at exact keyword matches. |
| **Vector (Neural)** | Embedding similarity search on `content_embedding` (1024-dim Titan V2). The query is embedded on-the-fly by the search Lambda. Excels at semantic/conceptual queries. |
| **Hybrid** | Combines both via the search pipeline: min_max normalization + arithmetic_mean with weights [0.3 BM25, 0.7 neural]. Aims to get the best of both worlds. |

## Prerequisites

- FastAPI server running on `localhost:8000`
- FAISS+BM25 search index populated with sagemaker-docs (336 documents)
- `pip install -r requirements.txt`

In [ ]:
import sys
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

# Add experiments/ to path so helpers can be imported
sys.path.insert(0, ".")
from helpers import (
    search,
    results_to_dataframe,
    hits_to_doc_ids,
    jaccard_similarity,
    overlap_matrix,
    rank_biased_overlap,
    BASE_URL,
)

# Verify the API is reachable
resp = requests.get(f"{BASE_URL}/search/index/stats", timeout=10)
stats = resp.json()
print(f"Index: {stats['index_name']}")
print(f"Documents: {stats['doc_count']}")
print(f"Status: {stats['status']}")

## Query Design

We use 9 queries across 3 categories, each designed to highlight where a particular search type should excel:

| Category | Rationale | Expected Winner |
|----------|-----------|----------------|
| **Exact technical terms** | BM25 matches exact tokens directly; vector embeddings may dilute rare terms across the embedding space | Text (BM25) |
| **Conceptual / semantic** | No exact token overlap with documents; embeddings capture meaning and intent | Vector (Neural) |
| **Mixed terms + concepts** | Contains both precise technical terms and conceptual framing that requires understanding | Hybrid |

In [ ]:
# Define the 9 test queries organized by category
QUERIES = {
    "text_excelling": [
        "RetainAllVariantProperties",
        "SAGEMAKER_NOTEBOOK_NO_DIRECT_INTERNET_ACCESS",
        "AmazonSageMakerFullAccess",
    ],
    "vector_excelling": [
        "How do I make sure my notebook isn't exposed to the internet?",
        "What is the benefit of using a project instead of running pipelines directly?",
        "How can data scientists share code consistently across a team?",
    ],
    "hybrid_excelling": [
        "What IAM permissions does an execution role need to run a training job?",
        "How does EventBridge trigger actions when an endpoint changes status?",
        "What kubectl commands do I use to check a training job running in Kubernetes?",
    ],
}

SEARCH_TYPES = ["text", "vector", "hybrid"]
RESULT_SIZE = 10

print(f"Total queries: {sum(len(v) for v in QUERIES.values())}")
print(f"Search types: {SEARCH_TYPES}")
print(f"Results per query per type: {RESULT_SIZE}")

In [ ]:
# Execute all queries across all search types
results = {}  # key: (category, query, search_type) -> response dict

for category, queries in QUERIES.items():
    for query in queries:
        for search_type in SEARCH_TYPES:
            print(f"  [{search_type:6s}] {query[:60]}..." if len(query) > 60 else f"  [{search_type:6s}] {query}")
            response = search(query, search_type=search_type, size=RESULT_SIZE)
            results[(category, query, search_type)] = response

print(f"\nTotal API calls: {len(results)}")

## Results: Text-Excelling Queries (Exact Technical Terms)

These queries use exact CloudFormation property names, AWS Config rule IDs, and IAM policy names. BM25 should find these via direct token matching, while vector search may struggle because rare technical identifiers get diluted in embedding space.

In [ ]:
for query in QUERIES["text_excelling"]:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    for search_type in SEARCH_TYPES:
        resp = results[("text_excelling", query, search_type)]
        df = results_to_dataframe(resp)
        print(f"\n--- {search_type.upper()} ({resp['total_hits']} total hits) ---")
        display(df)

## Results: Vector-Excelling Queries (Conceptual / Semantic)

These queries use natural language that describes a concept without using the exact terms found in the documents. Vector search should capture the semantic intent, while BM25 may miss because there is little token overlap.

In [ ]:
for query in QUERIES["vector_excelling"]:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    for search_type in SEARCH_TYPES:
        resp = results[("vector_excelling", query, search_type)]
        df = results_to_dataframe(resp)
        print(f"\n--- {search_type.upper()} ({resp['total_hits']} total hits) ---")
        display(df)

## Results: Hybrid-Excelling Queries (Mixed Terms + Concepts)

These queries contain both exact technical terms (anchoring the domain) and conceptual framing that requires semantic understanding. Hybrid search should benefit from both signals.

In [ ]:
for query in QUERIES["hybrid_excelling"]:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    for search_type in SEARCH_TYPES:
        resp = results[("hybrid_excelling", query, search_type)]
        df = results_to_dataframe(resp)
        print(f"\n--- {search_type.upper()} ({resp['total_hits']} total hits) ---")
        display(df)

## Overlap Analysis

How much do the result sets overlap between search types? Two metrics are used:

- **Jaccard Similarity**: Proportion of shared results regardless of ranking (set-based)
- **Rank-Biased Overlap (RBO)**: Position-weighted overlap — agreement at the top of the list matters more than at the bottom (p=0.9)

In [ ]:
# Per-query overlap matrices
all_queries = [q for qs in QUERIES.values() for q in qs]
all_categories = [cat for cat, qs in QUERIES.items() for _ in qs]

jaccard_matrices = []
rbo_data = []

for cat, query in zip(all_categories, all_queries):
    ids = {st: hits_to_doc_ids(results[(cat, query, st)]) for st in SEARCH_TYPES}
    jm = overlap_matrix(ids)
    jaccard_matrices.append(jm)

    # Pairwise RBO
    for i, a in enumerate(SEARCH_TYPES):
        for b in SEARCH_TYPES[i + 1:]:
            rbo_val = rank_biased_overlap(ids[a], ids[b])
            rbo_data.append({"query": query[:50], "pair": f"{a} vs {b}", "rbo": round(rbo_val, 3)})

# Average Jaccard matrix across all queries
avg_jaccard = sum(jaccard_matrices) / len(jaccard_matrices)
print("Average Jaccard Similarity Matrix (across all 9 queries):")
display(avg_jaccard.round(3))

In [ ]:
# Jaccard heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(avg_jaccard.values, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(SEARCH_TYPES)))
ax.set_yticks(range(len(SEARCH_TYPES)))
ax.set_xticklabels(SEARCH_TYPES)
ax.set_yticklabels(SEARCH_TYPES)
for i in range(len(SEARCH_TYPES)):
    for j in range(len(SEARCH_TYPES)):
        ax.text(j, i, f"{avg_jaccard.values[i, j]:.3f}", ha="center", va="center", fontsize=12)
plt.colorbar(im, ax=ax, label="Jaccard Similarity")
ax.set_title("Average Result Set Overlap (Jaccard)")
plt.tight_layout()
plt.show()

In [ ]:
# RBO comparison table
rbo_df = pd.DataFrame(rbo_data)
print("Rank-Biased Overlap (RBO, p=0.9) per query:")
display(rbo_df.pivot(index="query", columns="pair", values="rbo").round(3))

## Score Distribution Analysis

How do relevance scores distribute across search types and query categories?

In [ ]:
# Collect all scores by search type and category
score_data = []
for (cat, query, search_type), resp in results.items():
    for hit in resp.get("hits", []):
        score_data.append({
            "category": cat,
            "search_type": search_type,
            "score": hit["score"],
        })

score_df = pd.DataFrame(score_data)

# Box plot: score distribution by search type
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
categories = list(QUERIES.keys())
category_labels = ["Text-Excelling", "Vector-Excelling", "Hybrid-Excelling"]

for ax, cat, label in zip(axes, categories, category_labels):
    cat_df = score_df[score_df["category"] == cat]
    cat_df.boxplot(column="score", by="search_type", ax=ax)
    ax.set_title(label)
    ax.set_xlabel("Search Type")
    ax.set_ylabel("Score")

plt.suptitle("Score Distributions by Query Category", fontsize=14)
plt.tight_layout()
plt.show()

## Unique Documents Per Search Type

Which documents appear exclusively in one search type's results but not the others? This highlights the complementary value of each approach.

In [ ]:
for cat, query in zip(all_categories, all_queries):
    ids = {st: set(hits_to_doc_ids(results[(cat, query, st)])) for st in SEARCH_TYPES}
    all_ids = ids["text"] | ids["vector"] | ids["hybrid"]

    unique = {}
    for st in SEARCH_TYPES:
        others = set()
        for other_st in SEARCH_TYPES:
            if other_st != st:
                others |= ids[other_st]
        unique[st] = ids[st] - others

    if any(unique.values()):
        print(f"\nQuery: {query[:70]}")
        for st in SEARCH_TYPES:
            if unique[st]:
                print(f"  {st} only: {len(unique[st])} unique doc(s)")

## Observations and Takeaways

_Fill in after running the notebook with actual results._

### Questions to Answer

1. **Which search type produced the most relevant results for each query category?**
   - Text-excelling queries: _TODO_
   - Vector-excelling queries: _TODO_
   - Hybrid-excelling queries: _TODO_

2. **How much overlap exists between search types?**
   - Average Jaccard similarity: _TODO_
   - Are text and vector results largely disjoint or mostly overlapping?

3. **Are the current hybrid weights (0.3 BM25, 0.7 neural) appropriate?**
   - Does hybrid consistently outperform both individual types?
   - Would different weights improve results for certain query categories?

4. **Practical recommendation:**
   - _TODO: Should hybrid remain the default? Should we offer search type selection to users?_